# Task 2.2.8: Model Evaluation and Testing
## Testing the trained model on test image set and evaluating its performance

In [ ]:
# Your trained model weights should be saved from the training process.
# Load these weights into a YOLOv5 model to reconstruct your trained model.

import torch

# Load the trained YOLOv5 model
model = torch.hub.load('C:/Users/vasja/Desktop/AIM4VET/LM2TU2/yolo/yolov5/', 'custom', path='C:/Users/vasja/Desktop/AIM4VET/LM2TU2/yolo/yolov5/runs/train/my_first_model3/weights/best.pt', source='local')

In [ ]:
# Run each test image through the model and get the prediction.
# The prediction will contain the class labels and bounding box coordinates for each detected object.

import os

# Load the images that you have set aside for testing. These images were not seen by the model during the training process.
# Directory containing the test images
test_images_directory = "C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_testing/"

# Directory to save the results
output_directory = "C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_results1"

# Create output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Load and process each image in the directory
for img_name in os.listdir(test_images_directory):
    if img_name.endswith(('.jpg', '.jpeg', '.png')):  # checking if the file is an image
        # Perform inference
        img_path = os.path.join(test_images_directory, img_name)
        img = Image.open(img_path)
        results = model(img)
        
        # Get predictions
        # [class_index] [confidence] [x_center] [y_center] [width] [height]
        preds = results.pred[0]
        print(preds)
        
        # Save results as a text file
        output_file_path = os.path.join(output_directory, f"{os.path.splitext(img_name)[0]}_predictions.txt")
        with open(output_file_path, 'w') as file:
            for pred in preds:
                class_index, confidence, x_center, y_center, width, height = pred.tolist()
                # file.write(f"{class_index} {confidence} {x_center} {y_center} {width} {height}\n")
                file.write(f"{height} {width} {y_center} {x_center} {confidence} {class_index}\n")
        
        # Displaying results (optional)
        results.show()

### Evaluating our model: Evaluation Metrics

In [37]:
# First we need to normalize the predictions for each image, in order to make the data standard with the ground truth data.
# We should achieve the following data structure within our ground truth and prediction .txt files:
# Ground Truth .txt files: [class_id] [x_center] [y_center] [width] [height], where all the positional values are normalized for the image size.
# Prediction .txt files: [class_id] [confidence] [x_center] [y_center] [width] [height], where all the positional values are normalized for the image size.

from PIL import Image

def normalize_predictions(prediction_dir, image_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for filename in os.listdir(prediction_dir):
        if filename.endswith('_predictions.txt'):
            img_name = filename.replace('_predictions.txt', '.jpg')
            img_path = os.path.join(image_dir, img_name)
            
            with Image.open(img_path) as img:
                img_width, img_height = img.size

            filepath = os.path.join(prediction_dir, filename)
            output_filepath = os.path.join(output_dir, filename)
            
            with open(filepath, 'r') as file, open(output_filepath, 'w') as output_file:
                for line in file:
                    parts = line.strip().split()
                    values = [float(p) for p in parts]

                    class_id, confidence, y_max, x_max, y_min, x_min = values

                    x_center = (x_min + x_max) / 2 / img_width
                    y_center = (y_min + y_max) / 2 / img_height
                    width = abs(x_max - x_min) / img_width
                    height = abs(y_max - y_min) / img_height

                    new_line = f"{class_id} {confidence} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n"
                    output_file.write(new_line)

normalize_predictions('C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_results1', 'C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_testing', 'C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_results1_normalized')

#### Intersection over Union

In [40]:
# Defining a function to calculate the IoU between two bounding boxes accepted as parameters.

def calculate_iou(box1, box2):   
    # Extract the coordinates and dimensions from the input boxes
    x_center1, y_center1, width1, height1 = box1
    x_center2, y_center2, width2, height2 = box2
    
    # Calculate the coordinates of the corners of the boxes
    x1_min, y1_min = x_center1 - width1 / 2, y_center1 - height1 / 2
    x1_max, y1_max = x_center1 + width1 / 2, y_center1 + height1 / 2
    x2_min, y2_min = x_center2 - width2 / 2, y_center2 - height2 / 2
    x2_max, y2_max = x_center2 + width2 / 2, y_center2 + height2 / 2
    
    # Calculate the coordinates of the intersection rectangle
    x_min_int = max(x1_min, x2_min)
    y_min_int = max(y1_min, y2_min)
    x_max_int = min(x1_max, x2_max)
    y_max_int = min(y1_max, y2_max)
    
    # Calculate the area of intersection
    intersection_area = max(0, x_max_int - x_min_int) * max(0, y_max_int - y_min_int)
    
    # Calculate the area of each box
    box1_area = width1 * height1
    box2_area = width2 * height2
    
    # Calculate the area of union
    union_area = box1_area + box2_area - intersection_area
    
    # Calculate IoU
    iou = intersection_area / union_area if union_area > 0 else 0
    
    return iou

In [ ]:
import os

# Open all bounding boxes as lists of values.
def get_boxes_from_file(filepath):
    with open(filepath, 'r') as file:
        lines = file.readlines()
    
    boxes = []
    for line in lines:
        box = [float(value) for value in line.split()]
        boxes.append(box)
    
    return boxes

# Function to calculate the IoU for each prediction and also the total average IoU
def calculate_average_iou(prediction_dir, ground_truth_dir):
    total_iou = 0
    num_comparisons = 0
    
    for filename in os.listdir(prediction_dir):
        if filename.endswith('_predictions.txt'):
            pred_filepath = os.path.join(prediction_dir, filename)
            gt_filename = filename.replace('_predictions', '')
            gt_filepath = os.path.join(ground_truth_dir, gt_filename)
            
            pred_boxes = get_boxes_from_file(pred_filepath)
            gt_boxes = get_boxes_from_file(gt_filepath)
            
            for gt_box in gt_boxes:
                max_iou = 0
                for pred_box in pred_boxes:
                    iou = calculate_iou(pred_box[2:], gt_box[1:])
                    max_iou = max(max_iou, iou)
                    print("IoU for file ", gt_filename, " is: ", iou)
                
                print("MAX IoU for file ", gt_filename, " is: ", max_iou)
                total_iou += max_iou
                num_comparisons += 1
                
    average_iou = total_iou / num_comparisons if num_comparisons != 0 else 0
    return average_iou


prediction_dir = 'C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_results1_normalized'
ground_truth_dir = 'C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_testing'

average_iou = calculate_average_iou(prediction_dir, ground_truth_dir)
print(f"Average IoU: {average_iou}")

#### Precision and Recall

***Precision = True Positives (TP) / (False Positives (FP) + True Positives (TP)​)***  
***Recall = True Positives (TP) / (False Negatives (FN) + True Positives (TP)​)***

In [ ]:
# Defining function to calculate precision and recall, using the previous function `calculate_iou()` to calculate IoU.

def calculate_precision_recall(prediction_dir, ground_truth_dir, iou_threshold):
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    for filename in os.listdir(prediction_dir):
        if filename.endswith('_predictions.txt'):
            pred_filepath = os.path.join(prediction_dir, filename)
            gt_filename = filename.replace('_predictions', '')
            gt_filepath = os.path.join(ground_truth_dir, gt_filename)
            
            pred_boxes = get_boxes_from_file(pred_filepath)
            gt_boxes = get_boxes_from_file(gt_filepath)
            
            matched_gt_boxes = set()  # To keep track of matched ground truth boxes
            
            for pred_box in pred_boxes:
                max_iou = 0
                max_iou_index = -1
                
                for i, gt_box in enumerate(gt_boxes):
                    iou = calculate_iou(pred_box[2:], gt_box[1:])
                    if iou > max_iou:
                        max_iou = iou
                        max_iou_index = i
                
                if max_iou >= iou_threshold:
                    total_tp += 1
                    matched_gt_boxes.add(max_iou_index)
                else:
                    total_fp += 1
            
            # Counting unmatched ground truth boxes as false negatives
            total_fn += len(gt_boxes) - len(matched_gt_boxes)
            
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    
    return precision, recall

# Usage
threshold = 0.5
precision, recall = calculate_precision_recall(prediction_dir, ground_truth_dir, threshold)
print("Values for IoU threshold:", threshold)
print(f"Precision: {precision}")
print(f"Recall: {recall}")


#### mAP (mean Average Precision)

In [ ]:
# Calculating the Precision and Recall values at different IoU thresholds from 0.5 to 0.95 at 0.05 increments.

import numpy as np

def calculate_precision_recall_for_thresholds(prediction_dir, ground_truth_dir):
    # List to store tuples of precision and recall values for each IoU threshold
    precision_recall_pairs = []
    
    # Iterate through IoU thresholds from 0.5 to 0.95 with increments of 0.05
    thresholds = np.arange(0.5, 1.0, 0.05)
    
    for threshold in thresholds:
        precision, recall = calculate_precision_recall(prediction_dir, ground_truth_dir, threshold)
        
        # Saving precision and recall as a tuple in the list
        precision_recall_pairs.append((precision, recall))
        
        # Printing the values
        print(f"Values for IoU threshold {threshold:.2f}:")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print()  # Adding a newline for better readability
    
    # Returning the list of precision and recall pairs
    return precision_recall_pairs

# Calling the function and storing the result
precision_recall_pairs = calculate_precision_recall_for_thresholds(prediction_dir, ground_truth_dir)

In [ ]:
# Calculating mAP using precision and recall pairs obtained.

def calculate_map(precision_recall_pairs):
    # Sort pairs by recall values
    precision_recall_pairs.sort(key=lambda x: x[1])
    
    # Lists to store precision and recall values separately
    precision_list = [x[0] for x in precision_recall_pairs]
    recall_list = [x[1] for x in precision_recall_pairs]
    
    # Append values at the start and end to ensure the range of recall is complete
    precision_list = [0] + precision_list + [0]
    recall_list = [0] + recall_list + [1]
    
    # Update precision values to be the maximum of the precision to the right
    for i in range(len(precision_list) - 2, -1, -1):
        precision_list[i] = max(precision_list[i], precision_list[i+1])
        
    # Calculate the area under the precision-recall curve as the sum of rectangles
    ap = 0
    for i in range(1, len(recall_list)):
        if recall_list[i] != recall_list[i-1]:
            ap += precision_list[i] * (recall_list[i] - recall_list[i-1])
            
    return ap

# Calling the function by passing the precision_recall_pairs
map_value = calculate_map(precision_recall_pairs)

# Printing the mAP value
print(f"mAP: {map_value:.4f}")

### Task - Evaluating Trained Models With Tuned Hyperparameters

In the previous task you had to manually change the hyperparameters of the training, and observe the results of the training. Now, you need to calculate the IoU, Precision, Recall, and mAP values for a model trained with different hyperparameters.